In [ ]:
import duckdb


con = duckdb.connect("health_enforcement.duckdb")
con.execute("SHOW TABLES").df()


,name
0,dim_class_final
1,dim_fac_code
2,enriched_mart
3,fact_enforcement_actions
4,fact_ltc_narratives


In [6]:
# 1. Row counts — nothing should explode or disappear
print("Row Counts")
print(f"fact_enforcement_actions: {con.execute('SELECT COUNT(*) FROM fact_enforcement_actions').fetchone()[0]}")
print(f"fact_ltc_narratives:      {con.execute('SELECT COUNT(*) FROM fact_ltc_narratives').fetchone()[0]}")
print(f"enriched_mart:            {con.execute('SELECT COUNT(*) FROM enriched_mart').fetchone()[0]}")

# 2. enriched_mart should match fact_enforcement_actions exactly
print("mart and fact_enforcement")
mart_rows = con.execute("SELECT COUNT(*) FROM enriched_mart").fetchone()[0]
fact_rows = con.execute("SELECT COUNT(*) FROM fact_enforcement_actions").fetchone()[0]
print(f"Mart rows == Fact rows: {mart_rows == fact_rows} ({mart_rows} vs {fact_rows})")


Row Counts
fact_enforcement_actions: 20550
fact_ltc_narratives:      2885
enriched_mart:            20550
mart and fact_enforcement
Mart rows == Fact rows: True (20550 vs 20550)


In [7]:
df_mart = con.execute("SELECT * FROM enriched_mart").df()
print(df_mart.shape)
print(df_mart.dtypes)

(20550, 28)
EVENTID                                   str
FACID                                   int64
FACILITY_NAME                             str
FAC_TYPE_CODE                             str
EVENTID_1                                 str
FAC_FDR                                   str
DISTRICT_OFFICE                           str
LTC                                       str
FACILITY_TYPE_DESC                        str
IS_HOSPITAL                             int32
IS_24HR                                 int32
IS_LOW_RESOURCE                         int32
PENALTY_NUMBER                          int64
PENALTY_ISSUE_DATE             datetime64[us]
PENALTY_TYPE                              str
PENALTY_CATEGORY                          str
DISPOSITION                               str
CLASS_ASSESSED_INITIAL                    str
CLASS_ASSESSED_FINAL                      str
CLASS_ASSESSED_FINAL_DESC                 str
DEATH_RELATED                           int64
APPEALED              

In [16]:
import pandas as pd

In [1]:
con.close()

NameError: name 'con' is not defined

In [18]:
comparison = (df_mart.groupby(['CLASS_ASSESSED_FINAL', 'CLASS_ASSESSED_FINAL_DESC'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False))

print(comparison)

            CLASS_ASSESSED_FINAL     CLASS_ASSESSED_FINAL_DESC  count
8                              B                             B  11979
0                              A                             A   2735
15                        FTR AE                        FTR AE   1153
5                       AP NHPPD                      AP NHPPD    961
16                        FTR BR                        FTR BR    534
4                          AP IJ                         AP IJ    532
3                          AP BR                         AP BR    264
6                      AP NON-IJ                     AP NON-IJ    254
2                             AA                            AA    251
13               Drop>Deficiency               Drop>Deficiency    165
12            Dismissed by court            Dismissed by court    120
22                            WF                            WF     83
21                        UPHELD                        UPHELD     47
10                  